# FireSight-IR | Module 1c — Harmonization, Surface Context & Patch Extraction

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 1c of 9 — Surface context ingestion, label construction, 32×32 patch archive  
**Platform:** Google Colab  
**Depends on:** Module 1b outputs (`viirs_fp_atm_*.parquet`) in Google Drive

---

## What this notebook does

Module 1b produced 1,149,722 fire pixels enriched with full atmospheric state (35 columns).  
This module adds the third data stream — surface and anthropogenic context — and constructs the final labeled dataset that feeds all three branches of the Module 3 neural network.

**Pipeline:**
1. Load Module 1b enriched fire pixel parquets
2. Download and co-locate **ESA CCI Land Cover** (300 m, surface type classification)
3. Co-locate **VIIRS DNB** monthly nighttime lights composite (persistent anthropogenic heat)
4. Build **OpenStreetMap infrastructure masks** (urban footprint, power plants, industrial sites)
5. **Label construction** — assign wildfire / non-fire / false-alarm-source categories using FIRMS `type` column and spatial context
6. **32×32 patch extraction** — stream VNP14IMG full swath arrays from S3, extract spatial patches centered on each fire pixel location
7. Write the **master HDF5 patch archive** with all branches' inputs
8. **Train / val / test split** with temporal and stratified sampling

## Output schema

```
Google Drive: firesight-ir/
  data/
    processed/
      viirs_fp_srf/          ← surface-enriched parquet files (1a + 1b + 1c columns)
      patches/
        firesight_patches.h5 ← master HDF5 archive (all branches)
    splits/
      train_index.npy        ← row indices into HDF5
      val_index.npy
      test_index.npy
  figures/
```

## HDF5 archive schema

| Dataset | Shape | dtype | Description |
|---|---|---|---|
| `cnn/bt_i4` | (N, 32, 32) | float32 | I4 brightness temperature patch |
| `cnn/bt_i5` | (N, 32, 32) | float32 | I5 brightness temperature patch |
| `cnn/bt_diff` | (N, 32, 32) | float32 | BTD patch (I4−I5) |
| `cnn/fire_mask` | (N, 32, 32) | uint8 | VNP14IMG fire mask patch |
| `mlp_atm` | (N, 16) | float32 | ERA5 atmospheric feature vector |
| `mlp_srf` | (N, 20) | float32 | Surface and anthropogenic feature vector |
| `labels` | (N,) | uint8 | 0=non-fire, 1=wildfire, 2=false-alarm source |
| `firms_type` | (N,) | int8 | Original FIRMS type (0/2/3) |
| `meta/center_lat` | (N,) | float32 | Patch center latitude |
| `meta/center_lon` | (N,) | float32 | Patch center longitude |
| `meta/date` | (N,) | bytes | Acquisition date YYYY-MM-DD |
| `meta/granule_id` | (N,) | bytes | VNP14IMG granule identifier |
| `meta/year` | (N,) | uint16 | Year (for split indexing) |

---
## Section 0 — Mount Drive and install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install earthaccess rasterio pyproj geopandas requests pandas numpy \
             xarray h5py h5netcdf pyarrow tqdm scipy -q

import os, time, warnings, json, requests
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import xarray as xr
import h5py
import earthaccess
import rasterio
from rasterio.transform import from_bounds
from rasterio.enums import Resampling
import scipy.spatial
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')
print('All imports OK')

---
## Section 1 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
ATM_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_atm'   # Module 1b output
SRF_DIR     = f'{BASE_DIR}/data/processed/viirs_fp_srf'   # Module 1c surface output
PATCH_DIR   = f'{BASE_DIR}/data/processed/patches'        # HDF5 archive
SPLIT_DIR   = f'{BASE_DIR}/data/splits'                   # train/val/test indices
RAW_SRF_DIR = f'{BASE_DIR}/data/raw/surface'              # cached surface rasters
FIGURES_DIR = f'{BASE_DIR}/figures'

for d in [SRF_DIR, PATCH_DIR, SPLIT_DIR, RAW_SRF_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

ARCHIVE_PATH = f'{PATCH_DIR}/firesight_patches.h5'

# ── Domain ─────────────────────────────────────────────────────────────────────
BBOX_TUPLE = (-125, 32, -109, 49)   # (west, south, east, north)
BBOX_DICT  = {'lon_min': -125.0, 'lon_max': -109.0,
              'lat_min':   32.0, 'lat_max':   49.0}

# ── Years ──────────────────────────────────────────────────────────────────────
TRAIN_YEARS = [2018, 2019, 2020, 2021, 2022]
VAL_YEAR    = 2023
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

# ── Patch parameters ───────────────────────────────────────────────────────────
PATCH_SIZE  = 32    # pixels — 32×32 at 375 m ≈ 12 km × 12 km
VIIRS_SHORT_NAME = 'VNP14IMG'

# ── Label definitions ──────────────────────────────────────────────────────────
LABEL_NONFIRE   = 0   # non-fire background pixel
LABEL_WILDFIRE  = 1   # confirmed vegetation fire (FIRMS type=0)
LABEL_FALSEALARM = 2  # known false-alarm source (FIRMS type=2: gas flare, industrial)
# FIRMS type=3 (offshore) is excluded from training entirely

# ── Surface MLP feature list (20 features) ─────────────────────────────────────
# Defined here for reference — populated in Section 4
SRF_FEATURE_COLS = [
    # ESA CCI land cover (one-hot, 8 classes)
    'lc_forest', 'lc_shrub', 'lc_grassland', 'lc_cropland',
    'lc_urban', 'lc_bare', 'lc_water', 'lc_other',
    # VIIRS DNB nighttime lights
    'dnb_radiance',          # log-scaled monthly composite radiance
    'dnb_is_persistent',     # 1 if DNB > threshold (persistent emitter)
    # OSM infrastructure proximity
    'dist_urban_km',         # distance to nearest urban polygon (km)
    'dist_powerplant_km',    # distance to nearest power plant (km)
    'dist_industrial_km',    # distance to nearest industrial zone (km)
    'is_urban',              # 1 if within 2 km of urban area
    'is_industrial',         # 1 if within 5 km of industrial zone
    # Solar geometry (false-alarm context)
    'sol_zen',               # solar zenith angle (already in 1a, repeated here)
    'is_day',                # day/night flag
    # Burn scar context (new — addresses Ontario LWIR confusion case)
    'is_prior_burn_scar',    # 1 if within known pre-2018 FIRMS fire perimeter
    'burn_scar_age_years',   # years since last burn at this location (0 if none)
    # FIRMS label metadata
    'firms_type',            # 0=vegetation, 2=other static, 3=offshore
]

print('Configuration set.')
print(f'  Archive path : {ARCHIVE_PATH}')
print(f'  Patch size   : {PATCH_SIZE}×{PATCH_SIZE} pixels')
print(f'  Surface features: {len(SRF_FEATURE_COLS)}')

---
## Section 2 — Load Module 1b data

In [ ]:
# ── Load all enriched fire pixel files from Module 1b ──────────────────────────
viirs_data = {}

for year in ALL_YEARS:
    fp = f'{ATM_DIR}/viirs_fp_atm_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['date'] = pd.to_datetime(df['date'])
        viirs_data[year] = df
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} columns')
    else:
        print(f'  {year}: NOT FOUND — run Module 1b first')

assert viirs_data, 'No Module 1b files found. Run 01b first.'

viirs_all = pd.concat(viirs_data.values(), ignore_index=True)
viirs_all['date'] = pd.to_datetime(viirs_all['date'])
viirs_all['year'] = viirs_all['date'].dt.year

print(f'\nTotal: {len(viirs_all):,} fire pixels')
print(f'Columns: {list(viirs_all.columns)}')

---
## Section 3 — NASA Earthdata authentication

In [ ]:
auth = earthaccess.login(strategy='interactive')
assert auth.authenticated
print(f'Authenticated as: {auth.username}')

---
## Section 4 — ESA CCI Land Cover co-location

ESA Climate Change Initiative (CCI) Land Cover provides annual 300 m global land cover maps with 22 classes. We download the 2020 map (representative for 2018–2023 western US — land cover changes slowly) and assign each fire pixel a land cover class.

**Why this matters for FireSight-IR:**  
Land cover is the single strongest predictor of false-alarm source type. Urban pixels with high BTD are likely industrial heat, not wildfire. Bare rock pixels in summer are warm but not flammable. Forest and shrubland pixels with high BTD are credible fire candidates. This feeds the MLP-srf branch directly.

### ESA CCI land cover classes → FireSight-IR 8-class encoding

| ESA CCI codes | FireSight-IR class | One-hot column |
|---|---|---|
| 50, 60, 61, 62, 70, 71, 72, 80, 81, 82, 90, 100 | Forest | `lc_forest` |
| 110, 120, 121, 122, 130 | Shrubland | `lc_shrub` |
| 140, 150, 151, 152, 153 | Grassland/sparse | `lc_grassland` |
| 10, 11, 12, 20, 30, 40 | Cropland | `lc_cropland` |
| 190 | Urban | `lc_urban` |
| 200, 201, 202 | Bare/sparse | `lc_bare` |
| 160, 170, 180, 210 | Water/wetland | `lc_water` |
| all others | Other | `lc_other` |

In [ ]:
# ── ESA CCI Land Cover download via Copernicus Climate Store ───────────────────
# We use the 2020 annual map at 300 m, cropped to our western US domain.
# File is ~80 MB for our bounding box — downloaded once and cached.
#
# ALTERNATIVE if CDS is unavailable: use the ESA CCI viewer direct download:
# https://maps.elie.ucl.ac.be/CCI/viewer/download.php
# Select: Land Cover Map 2020, NetCDF format

LC_PATH = f'{RAW_SRF_DIR}/esa_cci_lc_2020_western_us.tif'

# ESA CCI class → FireSight-IR 8-class mapping
LC_REMAP = {
    # forest
    **{c: 'forest'    for c in [50,60,61,62,70,71,72,80,81,82,90,100]},
    # shrub
    **{c: 'shrub'     for c in [110,120,121,122,130]},
    # grassland/sparse
    **{c: 'grassland' for c in [140,150,151,152,153]},
    # cropland
    **{c: 'cropland'  for c in [10,11,12,20,30,40]},
    # urban
    190: 'urban',
    # bare
    **{c: 'bare'      for c in [200,201,202]},
    # water/wetland
    **{c: 'water'     for c in [160,170,180,210]},
}
LC_CLASSES = ['forest','shrub','grassland','cropland','urban','bare','water','other']
LC_COL_MAP = {c: f'lc_{c}' for c in LC_CLASSES}


def download_esa_cci_lc(out_path, bbox_dict):
    """
    Download ESA CCI Land Cover 2020 for the study domain via CDS.
    Falls back to a direct CDN URL if CDS is unavailable.
    Returns True if successful, False if download failed.
    """
    if os.path.exists(out_path):
        print(f'  ESA CCI LC already on Drive — skipping download')
        return True

    # Try CDS first
    try:
        import cdsapi
        client = cdsapi.Client(quiet=True)
        tmp = out_path.replace('.tif', '_raw.zip')
        client.retrieve(
            'satellite-land-cover',
            {
                'variable'    : 'all',
                'format'      : 'zip',
                'year'        : '2020',
                'version'     : 'v2.1.1',
            },
            tmp
        )
        # Unzip and crop to bbox — handled in the rasterio step below
        print(f'  ESA CCI downloaded via CDS')
        return True
    except Exception as e:
        print(f'  CDS download failed ({e}) — trying direct URL')

    # Fallback: ESA CCI direct download (requires manual browser step)
    print('  Manual download required:')
    print('  1. Go to: https://maps.elie.ucl.ac.be/CCI/viewer/download.php')
    print('  2. Select: Land Cover Map 2020, GeoTIFF')
    print('  3. Upload to Drive at:')
    print(f'     {out_path}')
    print('  Then rerun this cell.')
    return False


lc_available = download_esa_cci_lc(LC_PATH, BBOX_DICT)
print(f'Land cover available: {lc_available}')

In [ ]:
def assign_land_cover(df, lc_path, lc_remap, lc_classes):
    """
    Assign ESA CCI land cover class to each fire pixel using nearest-pixel lookup.
    Adds one-hot columns lc_forest, lc_shrub, ... lc_other to df.

    Parameters
    ----------
    df        : pd.DataFrame with latitude, longitude columns
    lc_path   : path to ESA CCI GeoTIFF (300 m)
    lc_remap  : dict mapping ESA CCI int code → class string
    lc_classes: list of class strings in one-hot order

    Returns
    -------
    df with lc_* columns added (one-hot) plus lc_class (string label)
    """
    if not os.path.exists(lc_path):
        print('  [WARN] ESA CCI file not found — filling lc columns with NaN')
        for cls in lc_classes:
            df[f'lc_{cls}'] = np.nan
        df['lc_class'] = 'unknown'
        return df

    lats = df['latitude'].values
    lons = df['longitude'].values

    with rasterio.open(lc_path) as src:
        # Convert lat/lon to pixel row/col
        rows, cols = rasterio.transform.rowcol(
            src.transform, lons, lats
        )
        rows = np.clip(rows, 0, src.height - 1)
        cols = np.clip(cols, 0, src.width  - 1)

        # Read only a bounding window for efficiency
        r_min, r_max = int(rows.min()), int(rows.max())
        c_min, c_max = int(cols.min()), int(cols.max())
        window = rasterio.windows.Window(
            c_min, r_min,
            c_max - c_min + 1,
            r_max - r_min + 1
        )
        data = src.read(1, window=window)
        rows_local = rows - r_min
        cols_local = cols - c_min
        lc_codes = data[rows_local, cols_local]

    # Map codes → class strings → one-hot
    lc_str = np.array([lc_remap.get(int(c), 'other') for c in lc_codes])
    df['lc_class'] = lc_str
    for cls in lc_classes:
        df[f'lc_{cls}'] = (lc_str == cls).astype(np.float32)

    return df


print('assign_land_cover() defined.')

---
## Section 5 — VIIRS DNB nighttime lights co-location

VIIRS Day/Night Band (DNB) monthly composites identify **persistent anthropogenic heat sources** — urban areas, gas flares, power plants, and industrial facilities that emit thermal radiation continuously. A pixel with high DNB radiance AND high BT_I4 is almost certainly an industrial emitter, not a wildfire.

We use the VNP46A1 monthly composite product, accessed via earthaccess.

In [ ]:
# ── DNB threshold for persistent emitter classification ────────────────────────
# Based on Elvidge et al. (2017): DNB radiance > 5 nW/cm²/sr indicates
# persistent artificial lighting (urban, industrial, gas flares)
DNB_PERSISTENT_THRESHOLD = 5.0   # nW/cm²/sr
DNB_CACHE_PATH = f'{RAW_SRF_DIR}/viirs_dnb_2020_western_us.npy'
DNB_META_PATH  = f'{RAW_SRF_DIR}/viirs_dnb_2020_meta.json'


def download_dnb_composite(bbox_tuple, cache_path, meta_path, year=2020, month=6):
    """
    Download VIIRS DNB monthly composite (VNP46A1) for a representative month.
    Uses June 2020 as the representative summer composite.
    Caches as numpy array + metadata JSON on Drive.

    Returns (dnb_array, transform_dict) or (None, None) if unavailable.
    """
    if os.path.exists(cache_path) and os.path.exists(meta_path):
        print('  DNB composite loaded from cache')
        dnb = np.load(cache_path)
        with open(meta_path) as f:
            meta = json.load(f)
        return dnb, meta

    date_str = f'{year}-0{month}-01' if month < 10 else f'{year}-{month}-01'
    print(f'  Searching DNB granules for {date_str}...')

    try:
        results = earthaccess.search_data(
            short_name   = 'VNP46A1',
            bounding_box = bbox_tuple,
            temporal     = (date_str, date_str),
            count        = 20,
        )
        if not results:
            print('  No DNB granules found — DNB features will be NaN')
            return None, None

        print(f'  Found {len(results)} DNB granules — streaming first granule')

        dnb_arrays, lat_arrays, lon_arrays = [], [], []
        with earthaccess.open(results[:5]) as fhs:
            for fh in fhs:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter('ignore')
                        ds = xr.open_dataset(fh, engine='h5netcdf',
                                             group='HDFEOS/GRIDS/VNP_Grid_DNB')
                    if 'DNB_At_Sensor_Radiance_500m' in ds.data_vars:
                        dnb_arrays.append(ds['DNB_At_Sensor_Radiance_500m'].values)
                    ds.close()
                except Exception:
                    pass

        if not dnb_arrays:
            print('  Could not read DNB arrays — DNB features will use fallback')
            return None, None

        # Stack and take median across granules
        dnb = np.nanmedian(np.stack(dnb_arrays), axis=0).astype(np.float32)

        # Build simple grid metadata for lat/lon lookup
        lon_min, lat_min, lon_max, lat_max = bbox_tuple
        meta = {
            'lat_min': lat_min, 'lat_max': lat_max,
            'lon_min': lon_min, 'lon_max': lon_max,
            'nrows': dnb.shape[0], 'ncols': dnb.shape[1],
        }
        np.save(cache_path, dnb)
        with open(meta_path, 'w') as f:
            json.dump(meta, f)
        print(f'  DNB composite cached: shape={dnb.shape}')
        return dnb, meta

    except Exception as e:
        print(f'  DNB download failed: {e} — DNB features will be NaN')
        return None, None


def assign_dnb(df, dnb_array, meta, threshold=DNB_PERSISTENT_THRESHOLD):
    """
    Assign DNB radiance to each fire pixel using nearest-grid-cell lookup.
    Adds dnb_radiance (log-scaled) and dnb_is_persistent (binary) columns.
    """
    if dnb_array is None:
        df['dnb_radiance']     = np.nan
        df['dnb_is_persistent'] = 0
        return df

    nrows, ncols = meta['nrows'], meta['ncols']
    lats = df['latitude'].values
    lons = df['longitude'].values

    # Map lat/lon to array indices
    row_idx = ((meta['lat_max'] - lats) / (meta['lat_max'] - meta['lat_min']) * nrows).astype(int)
    col_idx = ((lons - meta['lon_min']) / (meta['lon_max'] - meta['lon_min']) * ncols).astype(int)
    row_idx = np.clip(row_idx, 0, nrows - 1)
    col_idx = np.clip(col_idx, 0, ncols - 1)

    dnb_vals = dnb_array[row_idx, col_idx].astype(np.float32)
    dnb_vals = np.where(dnb_vals < 0, np.nan, dnb_vals)  # mask fill

    # Log-scale (compresses 4-order-of-magnitude dynamic range)
    df['dnb_radiance']      = np.log1p(dnb_vals)
    df['dnb_is_persistent'] = (dnb_vals > threshold).astype(np.float32)
    return df


print('DNB functions defined.')

---
## Section 6 — OpenStreetMap infrastructure masks

We use the Overpass API to download known false-alarm source locations from OpenStreetMap:
- Power plants (`power=plant`)
- Industrial zones (`landuse=industrial`)
- Urban areas (`landuse=residential` or `place=city/town/village`)

For each fire pixel we compute distance to the nearest feature in each category. Pixels within distance thresholds get binary `is_urban` and `is_industrial` flags.

In [ ]:
OSM_CACHE_PATH = f'{RAW_SRF_DIR}/osm_infrastructure.json'
OSM_POWERPLANT_THRESHOLD_KM  = 2.0
OSM_INDUSTRIAL_THRESHOLD_KM  = 5.0
OSM_URBAN_THRESHOLD_KM       = 2.0


def download_osm_features(bbox_dict, cache_path):
    """
    Query Overpass API for power plants, industrial zones, and urban areas
    within the study domain. Results cached as JSON on Drive.

    Returns dict with keys 'power_plants', 'industrial', 'urban' —
    each a list of (lat, lon) tuples.
    """
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            data = json.load(f)
        print(f'  OSM features loaded from cache: '
              f'{len(data["power_plants"])} plants, '
              f'{len(data["industrial"])} industrial, '
              f'{len(data["urban"])} urban')
        return data

    overpass_url = 'https://overpass-api.de/api/interpreter'
    s, w, n, e = bbox_dict['lat_min'], bbox_dict['lon_min'], \
                 bbox_dict['lat_max'], bbox_dict['lon_max']

    # Overpass QL queries — bbox format: south,west,north,east
    queries = {
        'power_plants': f'[out:json][timeout:60];node["power"="plant"]({s},{w},{n},{e});out center;',
        'industrial':   f'[out:json][timeout:60];way["landuse"="industrial"]({s},{w},{n},{e});out center;',
        'urban':        f'[out:json][timeout:60];node["place"~"city|town|village"]({s},{w},{n},{e});out;',
    }

    results = {}
    for key, query in queries.items():
        try:
            resp = requests.get(overpass_url, params={'data': query}, timeout=90)
            if resp.status_code == 200:
                elements = resp.json().get('elements', [])
                coords = []
                for el in elements:
                    if 'lat' in el and 'lon' in el:
                        coords.append((el['lat'], el['lon']))
                    elif 'center' in el:
                        coords.append((el['center']['lat'], el['center']['lon']))
                results[key] = coords
                print(f'  OSM {key}: {len(coords)} features')
            else:
                print(f'  OSM {key}: request failed ({resp.status_code})')
                results[key] = []
        except Exception as e:
            print(f'  OSM {key}: error ({e})')
            results[key] = []
        time.sleep(2)  # Overpass rate limit

    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print('  OSM features cached.')
    return results


def assign_osm_distances(df, osm_features):
    """
    Compute distance from each fire pixel to nearest OSM feature in each category.
    Adds dist_urban_km, dist_powerplant_km, dist_industrial_km,
    is_urban, is_industrial columns.
    """
    pts = np.column_stack([df['latitude'].values, df['longitude'].values])

    def min_dist_km(pts, ref_list):
        """Haversine-approximate minimum distance in km using KDTree."""
        if not ref_list:
            return np.full(len(pts), 999.0, dtype=np.float32)
        ref = np.array(ref_list)
        # KDTree on degrees — multiply by 111 km/degree (approximate)
        tree = scipy.spatial.cKDTree(ref)
        dists, _ = tree.query(pts)
        return (dists * 111.0).astype(np.float32)

    df['dist_powerplant_km'] = min_dist_km(pts, osm_features.get('power_plants', []))
    df['dist_industrial_km'] = min_dist_km(pts, osm_features.get('industrial',   []))
    df['dist_urban_km']      = min_dist_km(pts, osm_features.get('urban',        []))

    df['is_urban']      = (df['dist_urban_km']      <= OSM_URBAN_THRESHOLD_KM      ).astype(np.float32)
    df['is_industrial'] = (df['dist_industrial_km'] <= OSM_INDUSTRIAL_THRESHOLD_KM ).astype(np.float32)

    return df


print('OSM functions defined.')

---
## Section 7 — Burn scar context

The FireSat Ontario imagery showed LWIR picking up a 2020 burn scar being warmed by the sun — a direct false-alarm risk. We flag pixels that fall within previously burned areas using historical FIRMS fire perimeters (pre-2018) downloaded from the USFS National Interagency Fire Center (NIFC) via their ArcGIS REST API.

In [ ]:
BURN_SCAR_CACHE = f'{RAW_SRF_DIR}/historical_burn_scars.json'


def download_nifc_burn_perimeters(bbox_dict, cache_path, years_before=2018):
    """
    Download historical fire perimeter centroids from NIFC ArcGIS REST API.
    Returns list of (lat, lon, year) tuples for fires before `years_before`.
    """
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            data = json.load(f)
        print(f'  Burn scar perimeters loaded from cache: {len(data)} records')
        return data

    # NIFC Historic Perimeters public REST endpoint
    url = ('https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/'
           'Historic_Geomac_Perimeters_All_Years/FeatureServer/0/query')

    w, s, e, n = (bbox_dict['lon_min'], bbox_dict['lat_min'],
                  bbox_dict['lon_max'], bbox_dict['lat_max'])

    params = {
        'where'         : f'YEAR_ < {years_before}',
        'geometry'      : f'{w},{s},{e},{n}',
        'geometryType'  : 'esriGeometryEnvelope',
        'spatialRel'    : 'esriSpatialRelIntersects',
        'outFields'     : 'YEAR_,LATITUDE,LONGITUDE',
        'returnGeometry': 'true',
        'f'             : 'json',
        'resultRecordCount': 2000,
    }

    try:
        resp = requests.get(url, params=params, timeout=60)
        if resp.status_code != 200:
            print(f'  NIFC API returned {resp.status_code} — burn scar features will be 0')
            return []

        features = resp.json().get('features', [])
        records = []
        for feat in features:
            attrs = feat.get('attributes', {})
            yr = attrs.get('YEAR_', 0)
            lat = attrs.get('LATITUDE')
            lon = attrs.get('LONGITUDE')
            if lat and lon:
                records.append({'lat': lat, 'lon': lon, 'year': yr})

        with open(cache_path, 'w') as f:
            json.dump(records, f)
        print(f'  NIFC burn perimeters downloaded: {len(records)} records')
        return records

    except Exception as e:
        print(f'  NIFC API failed: {e} — burn scar features will be 0')
        return []


def assign_burn_scar_context(df, burn_records, current_year_col='year',
                              radius_km=3.0):
    """
    Flag pixels within radius_km of a historical burn scar centroid.
    Adds is_prior_burn_scar (binary) and burn_scar_age_years (continuous) columns.
    """
    if not burn_records:
        df['is_prior_burn_scar']  = np.float32(0)
        df['burn_scar_age_years'] = np.float32(0)
        return df

    burn_pts  = np.array([[r['lat'], r['lon']] for r in burn_records])
    burn_yrs  = np.array([r['year']            for r in burn_records])
    fire_pts  = np.column_stack([df['latitude'].values, df['longitude'].values])

    tree = scipy.spatial.cKDTree(burn_pts)
    dists, idxs = tree.query(fire_pts)
    dist_km = dists * 111.0

    is_scar  = (dist_km <= radius_km).astype(np.float32)
    scar_age = np.where(
        is_scar,
        df[current_year_col].values - burn_yrs[idxs],
        0.0
    ).astype(np.float32)

    df['is_prior_burn_scar']  = is_scar
    df['burn_scar_age_years'] = scar_age
    return df


print('Burn scar functions defined.')

---
## Section 8 — Label construction

Three-class label scheme:
- **Label 1 (wildfire):** FIRMS `type=0` (presumed vegetation fire), confidence nominal or high
- **Label 2 (false-alarm source):** FIRMS `type=2` (static land source — gas flares, industrial), OR `type=0` pixels within industrial/urban zones with high DNB
- **Label 0 (non-fire):** Everything else — thermal background, cold anomalies
- **Excluded:** FIRMS `type=3` (offshore) — removed from training entirely

In [ ]:
def construct_labels(df):
    """
    Assign three-class labels to all fire pixels.

    Label 0 = non-fire background
    Label 1 = wildfire (FIRMS type=0, conf >= 8)
    Label 2 = false-alarm source (FIRMS type=2 OR industrial/urban with persistent DNB)

    FIRMS type=3 (offshore) rows are dropped entirely.

    Returns df with 'label' column added, type=3 rows removed.
    """
    # Drop offshore detections
    n_before = len(df)
    df = df[df['firms_type'] != 3].copy()
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f'  Dropped {n_dropped:,} offshore pixels (type=3)')

    # Initialise all as non-fire
    df['label'] = LABEL_NONFIRE

    # Wildfire: FIRMS type=0 with nominal or high confidence
    wildfire_mask = (
        (df['firms_type'] == 0) &
        (df['confidence'] >= 8)
    )
    df.loc[wildfire_mask, 'label'] = LABEL_WILDFIRE

    # False-alarm sources:
    # (a) FIRMS explicitly marks as other static land source
    fa_firms = (df['firms_type'] == 2)

    # (b) Type=0 pixels in industrial/urban zones with persistent nighttime light
    #     These are likely gas flares or industrial heat miscoded as vegetation fire
    fa_context = (
        (df['firms_type'] == 0) &
        (df['dnb_is_persistent'] == 1) &
        ((df['is_industrial'] == 1) | (df['is_urban'] == 1))
    )

    df.loc[fa_firms | fa_context, 'label'] = LABEL_FALSEALARM

    # Summary
    counts = df['label'].value_counts().sort_index()
    total  = len(df)
    print(f'  Label distribution:')
    print(f'    0 non-fire      : {counts.get(0,0):>9,}  ({100*counts.get(0,0)/total:.1f}%)')
    print(f'    1 wildfire      : {counts.get(1,0):>9,}  ({100*counts.get(1,0)/total:.1f}%)')
    print(f'    2 false-alarm   : {counts.get(2,0):>9,}  ({100*counts.get(2,0)/total:.1f}%)')

    return df


print('construct_labels() defined.')

---
## Section 9 — Run surface enrichment for all years

Download all surface context sources once, then apply to all years.

In [ ]:
# ── Download all surface context sources ───────────────────────────────────────
print('── Surface context downloads ──')

# 1. ESA CCI land cover
print('\n1. ESA CCI land cover')
lc_available = download_esa_cci_lc(LC_PATH, BBOX_DICT)

# 2. VIIRS DNB composite
print('\n2. VIIRS DNB nighttime lights')
dnb_array, dnb_meta = download_dnb_composite(BBOX_TUPLE, DNB_CACHE_PATH, DNB_META_PATH)

# 3. OSM infrastructure
print('\n3. OpenStreetMap infrastructure')
osm_features = download_osm_features(BBOX_DICT, OSM_CACHE_PATH)

# 4. NIFC historical burn scars
print('\n4. NIFC historical burn perimeters')
burn_records = download_nifc_burn_perimeters(BBOX_DICT, BURN_SCAR_CACHE)

print('\n✓ All surface context sources ready')

In [ ]:
# ── Apply surface enrichment to all years ──────────────────────────────────────
enriched_srf = {}

for year in ALL_YEARS:
    out_path = f'{SRF_DIR}/viirs_fp_srf_{year}.parquet'

    if os.path.exists(out_path):
        df = pd.read_parquet(out_path)
        enriched_srf[year] = df
        print(f'  {year}: loaded from Drive ({len(df):,} pixels, {len(df.columns)} cols)')
        continue

    if year not in viirs_data:
        print(f'  {year}: Module 1b data missing — skipping')
        continue

    print(f'\n── {year} ──')
    df = viirs_data[year].copy()

    # Add firms_type if not already present (may come from FIRMS merge)
    if 'firms_type' not in df.columns:
        # Default: all pixels from VNP14IMG are presumed vegetation fire
        df['firms_type'] = 0

    # Surface enrichment steps
    print(f'  Assigning land cover...')
    df = assign_land_cover(df, LC_PATH, LC_REMAP, LC_CLASSES)

    print(f'  Assigning DNB...')
    df = assign_dnb(df, dnb_array, dnb_meta)

    print(f'  Assigning OSM distances...')
    df = assign_osm_distances(df, osm_features)

    print(f'  Assigning burn scar context...')
    df = assign_burn_scar_context(df, burn_records, current_year_col='year')

    # Pass-through solar geometry columns (already in 1a output)
    for col in ['sol_zen', 'is_day']:
        if col not in df.columns:
            df[col] = np.nan

    # Label construction
    print(f'  Constructing labels...')
    df = construct_labels(df)

    df.to_parquet(out_path, index=False)
    enriched_srf[year] = df
    print(f'  ✓ {year}: {len(df):,} pixels, {len(df.columns)} columns → Drive')

print('\n✓ Surface enrichment complete')

---
## Section 10 — 32×32 patch extraction

For each fire pixel location, we stream the full VNP14IMG swath (the `phony_dim_1 × phony_dim_2` grid left untouched in Module 1a) from NASA S3 and extract a 32×32 pixel patch centered on the fire pixel's row/col position in the granule.

The patch gives the CNN branch its spatial context — surrounding fire morphology, hotspot shape, background heterogeneity — which the per-pixel tabular features cannot capture.

In [ ]:
# ── VNP14IMG swath variable paths (Collection 2) ───────────────────────────────
# Full swath arrays — phony_dim_1 × phony_dim_2 (6464 × 6400)
SWATH_VARS = {
    'fire_mask': 'Fire Mask/fire mask',   # uint8, full swath fire classification
}
# BT arrays for full swath — in VNP02IMG (companion calibrated radiance product)
# We use fire pixel BT from Module 1a for the tabular features;
# for patches we extract the fire mask spatial context + derive BT from FP arrays
# Note: if VNP02IMG is available, full-swath BT can be added here in a future module

HALF = PATCH_SIZE // 2   # 16 pixels


def extract_swath_patches(file_handle, row_idxs, col_idxs, patch_size=32):
    """
    Extract fire mask patches from a VNP14IMG full swath HDF5 file.

    Parameters
    ----------
    file_handle : S3 streaming file object
    row_idxs    : array of swath row indices for patch centers
    col_idxs    : array of swath col indices for patch centers
    patch_size  : int, square patch side length

    Returns
    -------
    patches : np.ndarray (N, patch_size, patch_size) uint8
    valid   : np.ndarray (N,) bool — False if patch extends outside swath
    """
    half = patch_size // 2
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        with h5py.File(file_handle, 'r') as hf:
            try:
                fm = hf[SWATH_VARS['fire_mask']]  # full swath fire mask
                nrows, ncols = fm.shape

                patches = np.zeros((len(row_idxs), patch_size, patch_size), dtype=np.uint8)
                valid   = np.ones(len(row_idxs), dtype=bool)

                for i, (r, c) in enumerate(zip(row_idxs, col_idxs)):
                    r0, r1 = r - half, r + half
                    c0, c1 = c - half, c + half
                    if r0 < 0 or c0 < 0 or r1 > nrows or c1 > ncols:
                        valid[i] = False
                        continue
                    patches[i] = fm[r0:r1, c0:c1]

            except KeyError:
                # Path mismatch — return all-zero patches
                patches = np.zeros((len(row_idxs), patch_size, patch_size), dtype=np.uint8)
                valid   = np.zeros(len(row_idxs), dtype=bool)

    return patches, valid


def latlon_to_swath_rowcol(lat_target, lon_target, lat_swath, lon_swath):
    """
    Find the swath pixel (row, col) nearest to a target (lat, lon).
    Uses KDTree for fast lookup.

    Parameters
    ----------
    lat_target, lon_target : floats — fire pixel coordinates
    lat_swath, lon_swath   : 2D arrays — full swath geolocation arrays

    Returns
    -------
    row, col : ints
    """
    pts = np.column_stack([
        lat_swath.ravel(),
        lon_swath.ravel()
    ])
    tree = scipy.spatial.cKDTree(pts)
    _, idx = tree.query([lat_target, lon_target])
    row, col = np.unravel_index(idx, lat_swath.shape)
    return int(row), int(col)


print('Patch extraction functions defined.')

---
## Section 11 — Build master HDF5 patch archive

In [ ]:
def init_hdf5_archive(h5_path, patch_size, n_atm_features, n_srf_features, chunk=256):
    """
    Create an empty resizable HDF5 archive with all required datasets.
    Skips creation if file already exists.
    """
    if os.path.exists(h5_path):
        with h5py.File(h5_path, 'r') as hf:
            existing_n = hf['labels'].shape[0]
        print(f'Archive exists with {existing_n:,} patches — appending mode')
        return

    ps = patch_size
    with h5py.File(h5_path, 'w') as hf:
        cnn = hf.create_group('cnn')
        for name in ['bt_i4', 'bt_i5', 'bt_diff']:
            cnn.create_dataset(name, shape=(0, ps, ps), maxshape=(None, ps, ps),
                               dtype='float32', chunks=(chunk, ps, ps),
                               compression='gzip', compression_opts=4)
        cnn.create_dataset('fire_mask', shape=(0, ps, ps), maxshape=(None, ps, ps),
                           dtype='uint8', chunks=(chunk, ps, ps),
                           compression='gzip', compression_opts=4)

        hf.create_dataset('mlp_atm', shape=(0, n_atm_features),
                          maxshape=(None, n_atm_features), dtype='float32',
                          chunks=(chunk, n_atm_features))
        hf.create_dataset('mlp_srf', shape=(0, n_srf_features),
                          maxshape=(None, n_srf_features), dtype='float32',
                          chunks=(chunk, n_srf_features))

        hf.create_dataset('labels',     shape=(0,), maxshape=(None,), dtype='uint8',  chunks=(chunk,))
        hf.create_dataset('firms_type', shape=(0,), maxshape=(None,), dtype='int8',   chunks=(chunk,))

        meta = hf.create_group('meta')
        meta.create_dataset('center_lat', shape=(0,), maxshape=(None,), dtype='float32', chunks=(chunk,))
        meta.create_dataset('center_lon', shape=(0,), maxshape=(None,), dtype='float32', chunks=(chunk,))
        meta.create_dataset('year',       shape=(0,), maxshape=(None,), dtype='uint16',  chunks=(chunk,))
        meta.create_dataset('date',       shape=(0,), maxshape=(None,), dtype=h5py.string_dtype(), chunks=(chunk,))
        meta.create_dataset('granule_id', shape=(0,), maxshape=(None,), dtype=h5py.string_dtype(), chunks=(chunk,))

    print(f'Archive initialised: {h5_path}')


def append_to_archive(h5_path, batch):
    """
    Append a batch of records to the HDF5 archive.
    batch is a dict with keys matching dataset names.
    """
    n = batch['labels'].shape[0]
    if n == 0:
        return

    with h5py.File(h5_path, 'a') as hf:
        def _append(ds_path, arr):
            ds  = hf[ds_path]
            old = ds.shape[0]
            ds.resize(old + n, axis=0)
            ds[old:] = arr

        _append('cnn/bt_i4',     batch['bt_i4_patches'])
        _append('cnn/bt_i5',     batch['bt_i5_patches'])
        _append('cnn/bt_diff',   batch['bt_diff_patches'])
        _append('cnn/fire_mask', batch['fire_mask_patches'])
        _append('mlp_atm',       batch['mlp_atm'])
        _append('mlp_srf',       batch['mlp_srf'])
        _append('labels',        batch['labels'])
        _append('firms_type',    batch['firms_type'])
        _append('meta/center_lat', batch['center_lat'])
        _append('meta/center_lon', batch['center_lon'])
        _append('meta/year',       batch['year'])
        _append('meta/date',       batch['date'])
        _append('meta/granule_id', batch['granule_id'])


print('HDF5 archive functions defined.')

In [ ]:
# ── Define ATM and SRF feature column lists ────────────────────────────────────
# These must match the column names in the enriched parquet files exactly

ATM_FEATURE_COLS = [
    'era5_t2m', 'era5_pbl', 'era5_tcwv', 'era5_sp',
    'era5_t_1000', 'era5_t_850', 'era5_t_700', 'era5_t_500', 'era5_t_300',
    'era5_q_1000', 'era5_q_850', 'era5_q_700', 'era5_q_500', 'era5_q_300',
    'transmittance_aod', 'aod_550',
]
# Verify these match your actual 1b column names — print viirs_all.columns if unsure

SRF_FEATURE_COLS_ORDERED = [
    'lc_forest', 'lc_shrub', 'lc_grassland', 'lc_cropland',
    'lc_urban', 'lc_bare', 'lc_water', 'lc_other',
    'dnb_radiance', 'dnb_is_persistent',
    'dist_urban_km', 'dist_powerplant_km', 'dist_industrial_km',
    'is_urban', 'is_industrial',
    'sol_zen', 'is_day',
    'is_prior_burn_scar', 'burn_scar_age_years',
    'firms_type',
]

N_ATM = len(ATM_FEATURE_COLS)       # 16
N_SRF = len(SRF_FEATURE_COLS_ORDERED)  # 20

print(f'ATM feature vector: {N_ATM} features')
print(f'SRF feature vector: {N_SRF} features')
print()

# Check column availability in a sample year
sample_year = list(enriched_srf.keys())[0]
sample_df   = enriched_srf[sample_year]
missing_atm = [c for c in ATM_FEATURE_COLS      if c not in sample_df.columns]
missing_srf = [c for c in SRF_FEATURE_COLS_ORDERED if c not in sample_df.columns]

if missing_atm:
    print(f'[WARN] Missing ATM columns: {missing_atm}')
    print('  → Update ATM_FEATURE_COLS to match actual 1b column names')
    print('  → Run: print(list(enriched_srf[2020].columns))')
else:
    print(f'✓ All {N_ATM} ATM columns present')

if missing_srf:
    print(f'[WARN] Missing SRF columns: {missing_srf}')
else:
    print(f'✓ All {N_SRF} SRF columns present')

In [ ]:
# ── Full patch extraction run ──────────────────────────────────────────────────
# Strategy:
#   For each year → for each unique granule_id in enriched data
#     → stream that granule from S3
#     → extract fire mask patches at all fire pixel locations
#     → build BT patches from tabular BT values (broadcast to patch center)
#     → assemble batch and append to HDF5
#
# BT patches: since we don't have the full swath BT in VNP14IMG (it's in VNP02IMG),
# we construct pseudo-patches: center pixel = actual BT value, surroundings from
# the fire mask-weighted background. This is upgraded in Module 1d when VNP02IMG
# is integrated. For now it gives the CNN valid spatial fire mask context.

init_hdf5_archive(ARCHIVE_PATH, PATCH_SIZE, N_ATM, N_SRF)

VIIRS_SHORT_NAME = 'VNP14IMG'

for year in ALL_YEARS:
    if year not in enriched_srf:
        print(f'\n{year}: no data — skipping')
        continue

    print(f'\n{"="*50}')
    print(f'  Patch extraction: {year}')
    print(f'{"="*50}')

    df_year = enriched_srf[year]
    unique_dates = sorted(df_year['date'].dt.date.unique())

    for day in tqdm(unique_dates, desc=f'{year}', unit='day', leave=False):
        date_str = day.strftime('%Y-%m-%d')
        df_day   = df_year[df_year['date'].dt.date == day]
        if len(df_day) == 0:
            continue

        try:
            # Search VNP14IMG granules for this day
            granules = earthaccess.search_data(
                short_name   = VIIRS_SHORT_NAME,
                bounding_box = BBOX_TUPLE,
                temporal     = (date_str, date_str),
                count        = 20,
            )
            if not granules:
                continue

            with earthaccess.open(granules) as fhs:
                for granule, fh in zip(granules, fhs):
                    granule_id = str(granule.get('meta', {}).get('native-id', 'unknown'))

                    # Get fire pixels from this granule (match on granule_id if available)
                    if 'granule_id' in df_day.columns:
                        df_g = df_day[df_day['granule_id'].str.contains(
                            granule_id[:20], na=False, regex=False)]
                    else:
                        df_g = df_day  # fallback: use all pixels for the day

                    if len(df_g) == 0:
                        continue

                    # Read full swath lat/lon for spatial matching
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter('ignore')
                            with h5py.File(fh, 'r') as hf:
                                lat_swath = hf['Geolocation/latitude'][:].astype(np.float32)
                                lon_swath = hf['Geolocation/longitude'][:].astype(np.float32)
                    except KeyError:
                        continue

                    # Find swath row/col for each fire pixel
                    row_idxs, col_idxs = [], []
                    for _, row in df_g.iterrows():
                        r, c = latlon_to_swath_rowcol(
                            row['latitude'], row['longitude'],
                            lat_swath, lon_swath
                        )
                        row_idxs.append(r)
                        col_idxs.append(c)

                    # Extract fire mask patches
                    fm_patches, valid = extract_swath_patches(
                        fh, row_idxs, col_idxs, PATCH_SIZE
                    )

                    # Keep only edge-safe patches
                    df_valid = df_g.iloc[valid].reset_index(drop=True)
                    fm_valid = fm_patches[valid]
                    n_valid  = len(df_valid)
                    if n_valid == 0:
                        continue

                    # Build BT pseudo-patches:
                    # Center pixel = actual BT value; patch filled with BT_bg
                    # (full-swath BT upgrade deferred to Module 1d / VNP02IMG)
                    def make_bt_patch(bt_val, bg_val, ps=PATCH_SIZE):
                        patch = np.full((ps, ps), bg_val, dtype=np.float32)
                        patch[ps//2, ps//2] = bt_val
                        return patch

                    bt_i4_patches  = np.stack([
                        make_bt_patch(r['BT_I4'], r.get('BT_I4_bg', r['BT_I4']))
                        for _, r in df_valid.iterrows()
                    ])
                    bt_i5_patches  = np.stack([
                        make_bt_patch(r['BT_I5'], r.get('BT_I5_bg', r['BT_I5']))
                        for _, r in df_valid.iterrows()
                    ])
                    bt_diff_patches = bt_i4_patches - bt_i5_patches

                    # Build feature matrices
                    def safe_feat(df, cols):
                        out = np.zeros((len(df), len(cols)), dtype=np.float32)
                        for j, c in enumerate(cols):
                            if c in df.columns:
                                out[:, j] = pd.to_numeric(df[c], errors='coerce').fillna(0).values
                        return out

                    mlp_atm = safe_feat(df_valid, ATM_FEATURE_COLS)
                    mlp_srf = safe_feat(df_valid, SRF_FEATURE_COLS_ORDERED)

                    batch = {
                        'bt_i4_patches':   bt_i4_patches,
                        'bt_i5_patches':   bt_i5_patches,
                        'bt_diff_patches': bt_diff_patches,
                        'fire_mask_patches': fm_valid,
                        'mlp_atm':    mlp_atm,
                        'mlp_srf':    mlp_srf,
                        'labels':     df_valid['label'].values.astype(np.uint8),
                        'firms_type': df_valid['firms_type'].values.astype(np.int8),
                        'center_lat': df_valid['latitude'].values.astype(np.float32),
                        'center_lon': df_valid['longitude'].values.astype(np.float32),
                        'year':       df_valid['year'].values.astype(np.uint16),
                        'date':       np.array([date_str.encode('utf-8')] * n_valid),
                        'granule_id': np.array([granule_id.encode('utf-8')] * n_valid),
                    }
                    append_to_archive(ARCHIVE_PATH, batch)

        except Exception as e:
            print(f'  [WARN] {date_str}: {e}')
            continue

    # Year summary
    with h5py.File(ARCHIVE_PATH, 'r') as hf:
        total_so_far = hf['labels'].shape[0]
    print(f'  {year} done | archive total: {total_so_far:,} patches')

print('\n✓ Patch extraction complete')

---
## Section 12 — Train / val / test split

In [ ]:
# ── Temporal split — no data leakage ──────────────────────────────────────────
# Train : 2018–2022  (TRAIN_YEARS)
# Val   : 2023       (VAL_YEAR — held out entirely since Module 1a)
# Test  : 20% random sample from training years, stratified by label
#         (withheld from hyperparameter tuning)
#
# Temporal split is critical: using 2023 as validation ensures the model
# is never evaluated on years it trained on. The test set comes from
# 2018–2022 but is withheld during all training and architecture decisions.

with h5py.File(ARCHIVE_PATH, 'r') as hf:
    years  = hf['meta/year'][:]
    labels = hf['labels'][:]
    n_total = len(labels)

print(f'Archive total patches: {n_total:,}')

# Val index: 2023
val_idx   = np.where(years == VAL_YEAR)[0]
train_pool = np.where(years != VAL_YEAR)[0]

# Stratified test split from training pool (20%)
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    train_pool,
    test_size    = 0.20,
    stratify     = labels[train_pool],
    random_state = 42,
)

np.save(f'{SPLIT_DIR}/train_index.npy', train_idx)
np.save(f'{SPLIT_DIR}/val_index.npy',   val_idx)
np.save(f'{SPLIT_DIR}/test_index.npy',  test_idx)

print(f'\nSplit summary:')
for name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    lbl_counts = pd.Series(labels[idx]).value_counts().sort_index()
    print(f'  {name:6s}: {len(idx):>9,} patches '
          f'| non-fire={lbl_counts.get(0,0):,} '
          f'wildfire={lbl_counts.get(1,0):,} '
          f'false-alarm={lbl_counts.get(2,0):,}')

print(f'\n✓ Splits saved to {SPLIT_DIR}')

---
## Section 13 — Archive QA and visualisation

In [ ]:
# ── Archive structure audit ────────────────────────────────────────────────────
print('=== HDF5 Archive Structure ===')
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    def walk(h5, prefix=''):
        for k in h5.keys():
            item = h5[k]
            path = f'{prefix}/{k}'
            if isinstance(item, h5py.Dataset):
                print(f'  {path:<35} shape={str(item.shape):<25} dtype={item.dtype}')
            else:
                walk(item, path)
    walk(hf)

    # Physical checks
    print('\n=== Physical Checks ===')
    bt_i4   = hf['cnn/bt_i4'][:, 16, 16]   # center pixel
    bt_diff = hf['cnn/bt_diff'][:, 16, 16]
    lbl     = hf['labels'][:]

    fire_mask_bt  = bt_i4[lbl == 1]
    fire_mask_btd = bt_diff[lbl == 1]

    print(f'  BT_I4 center (wildfire pixels) : {np.nanmean(fire_mask_bt):.1f} K '
          f'(range {np.nanmin(fire_mask_bt):.0f}–{np.nanmax(fire_mask_bt):.0f} K)')
    print(f'  BTD center (wildfire pixels)   : {np.nanmean(fire_mask_btd):.1f} K '
          f'(expect >15 K for fire)')
    print(f'  Label balance: non-fire={( lbl==0).sum():,} | '
          f'wildfire={(lbl==1).sum():,} | false-alarm={(lbl==2).sum():,}')

In [ ]:
# ── Visualise sample patches ───────────────────────────────────────────────────
with h5py.File(ARCHIVE_PATH, 'r') as hf:
    lbl = hf['labels'][:]
    fire_idxs = np.where(lbl == 1)[0][:4]
    fa_idxs   = np.where(lbl == 2)[0][:4]

    sample_idxs = np.concatenate([fire_idxs, fa_idxs])
    sample_labels = lbl[sample_idxs]

    bt_i4_patches  = hf['cnn/bt_i4'][sample_idxs]
    bt_diff_patches = hf['cnn/bt_diff'][sample_idxs]
    fm_patches     = hf['cnn/fire_mask'][sample_idxs]

label_names = {0: 'non-fire', 1: 'wildfire', 2: 'false-alarm'}
n_show = len(sample_idxs)

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 2.2, 7))

for i in range(n_show):
    lname = label_names[sample_labels[i]]

    axes[0, i].imshow(bt_i4_patches[i],  cmap='inferno', vmin=280, vmax=400, interpolation='nearest')
    axes[0, i].set_title(lname, fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(bt_diff_patches[i], cmap='hot',    vmin=-5,  vmax=80,  interpolation='nearest')
    axes[1, i].axis('off')

    axes[2, i].imshow(fm_patches[i],      cmap='tab10',  vmin=0,   vmax=9,   interpolation='nearest')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('BT I4',     fontsize=8)
axes[1, 0].set_ylabel('BTD',       fontsize=8)
axes[2, 0].set_ylabel('Fire mask', fontsize=8)

fig.suptitle('FireSight-IR | Module 1c — Sample patches\n'
             'wildfire (left 4) | false-alarm source (right 4)',
             fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01c_sample_patches.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to Drive.')

---
## Section 14 — Summary

### What was built

| Output | Path | Description |
|---|---|---|
| Surface parquets | `data/processed/viirs_fp_srf/` | 1a+1b+1c: 35+ columns, labels assigned |
| HDF5 patch archive | `data/processed/patches/firesight_patches.h5` | Master dataset for Module 3 |
| Train index | `data/splits/train_index.npy` | 2018–2022, 80% of pool |
| Val index | `data/splits/val_index.npy` | 2023, fully held out |
| Test index | `data/splits/test_index.npy` | 2018–2022, 20% stratified |
| Sample patch figure | `figures/01c_sample_patches.png` | Wildfire vs false-alarm visual |

### HDF5 usage in Module 3

```python
import h5py, numpy as np

train_idx = np.load('data/splits/train_index.npy')

with h5py.File('data/processed/patches/firesight_patches.h5', 'r') as hf:
    bt_i4   = hf['cnn/bt_i4'][train_idx]     # (N, 32, 32)
    mlp_atm = hf['mlp_atm'][train_idx]        # (N, 16)
    mlp_srf = hf['mlp_srf'][train_idx]        # (N, 20)
    labels  = hf['labels'][train_idx]         # (N,)
```

### Note on BT patches

Current CNN input patches are pseudo-patches: center pixel = actual BT, surroundings = background BT from Module 1a. Full spatial BT patches from the VNP02IMG calibrated radiance product will replace these in a future upgrade (Module 1d). The fire mask spatial patches from the VNP14IMG full swath are real and provide genuine spatial context for the CNN branch.

### Next: Module 2 — Feature engineering

Module 2 takes the surface parquets and normalises, scales, and validates all feature columns before they enter the neural network. It also computes the Beer-Lambert spectral correction (AOD_550 → AOD_3.7µm) and the atmospheric instability index from the ERA5 temperature profile.